# 📈 Zeitreihenprognose mit Prophet
## Hands-On Crashkurs

---

In dieser Session lernen wir, wie Prophet unter der Haube funktioniert. Statt mit echten, unaufgeräumten Daten zu arbeiten, generieren wir uns direkt einen **fertigen, synthetischen Umsatz-Datensatz** (in USD) für ein fiktives E-Commerce-Produkt (SkyDrive X1 Pro SSD).

Da wir die Struktur der Daten selbst definieren, können wir am Ende exakt überprüfen, ob Prophet die eingebauten Muster korrekt identifiziert.

### 🧮 Die mathematische Basis (Die Prophet-Gleichung)
Prophet modelliert Zeitreihen als additives Modell, das sich aus drei Hauptkomponenten und einem Fehlerterm zusammensetzt:

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

- **$g(t)$:** Trend (langfristige Veränderungen wie Wachstum/Rückgang)
- **$s(t)$:** Saisonalität (periodische Veränderungen, z. B. wöchentlich oder jährlich)
- **$h(t)$:** Holidays / Events (Ausreißer durch Feiertage oder besondere Ereignisse)
- **$\epsilon_t$:** Fehlerterm (zufälliges Rauschen)

Wir werden in diesem Beispiel besonders den Effekt des **Holiday Features ($h(t)$)** kennenlernen und herausfinden, warum Domänenwissen bei Ausreißern unverzichtbar ist. 

**Unser synthetischer Umsatz-Datensatz enthält exakt diese Bausteine:**
- 📈 Einen stetigen **Aufwärtstrend** (Umsatzwachstum über die Jahre)
- 📅 **Jährliche Saisonalität:** Höhere Umsätze im Sommer
- 📆 **Wöchentliche Saisonalität:** Umsatzspitzen an den Wochenenden
- 🛍️ **Black Friday:** Ein starker Ausreißer jeden November

Das Notebook ist dreigeteilt. Im ersten Teil wird der Datensatz gemeinsam erzeugt. Im zweiten soll PROPHET zunächst ohne das holidays feature ausprobiert werden. Im dritten Teil schließlich in der analyst-in-the-loop Variante mit dem Expertenwissen zum Black Friday.

---
## 📊 Unser Datensatz – SkyDrive X1 Pro externe SSD

Wir simulieren den **täglichen Umsatz** einer externen SSD – der *SkyDrive X1 Pro* –
über 4 Jahre (2020–2023). Das Produkt profitiert jedes Jahr stark vom **Black Friday**
und den umliegenden Tagen.

**Preisstruktur:**

| Zeitraum | Verkaufspreis | Rabatt | Mengen-Effekt |
|---|---|---|---|
| Normaltage | $129.00 | – | Basismenge (Trend + Saisonalität) |
| 3 Tage vor Black Friday | $104.49 | −19 % | +120 Einheiten/Tag (Vorverkauf) |
| Black Friday | $78.69 | −39 % | +500 Einheiten (Haupt-Event) |
| Tag nach Black Friday (Sa) | $78.69 | −39 % | +250 Einheiten (Angebote laufen weiter) |

So sehen die Umsatzdaten aus – die wir im Folgenden synthetisch nachbauen:

![Alt-Text](umsaetzeSkyDive_v2.png)

---
## 🔧 Teil A: Setup & Datengenerierung (vorgegeben)

Dieser Teil ist bereits fertig – führe die Zellen einfach aus.

In [1]:
# --- DATENGENERIERUNG (Komplett vorgegeben) ---
import pandas as pd
import numpy as np

np.random.seed(42)
dates = pd.date_range(start='2020-01-01', end='2023-12-31')
df = pd.DataFrame({'ds': dates})  # Prophet erwartet die Datumsspalte immer als 'ds'

# ── 1. Basis-Umsatzbausteine (direkt in USD $) ───────────────────────────────
# Trend: linearer Umsatzanstieg von ~6.450 $ auf 19.350 $ pro Tag
df['trend'] = np.linspace(6450, 19350, len(dates))

# Rauschen: unvorhersehbare Schwankungen (σ = 650 $)
df['noise'] = np.random.normal(0, 650, len(dates))

# Wöchentliche Saisonalität (Umsatz-Abweichung vom Tagesmittel in $)
df['weekly'] = df['ds'].dt.dayofweek.map({
    0: -1030, 1: -1290, 2: -1290, 3: -1030, 4: 650, 5: 2580, 6: 1935
})

# Jährliche Saisonalität: Amplitude ±3.225 $ (Sommer vs. Winter)
df['yearly'] = 3225 * np.sin(2 * np.pi * (df['ds'].dt.dayofyear - 100) / 365.25)


### 🛍️ Was ist der Black Friday – und wann fällt er?

Der Black Friday findet traditionell immer am Tag **nach dem amerikanischen Feiertag Thanksgiving** statt.
Da Thanksgiving in den USA immer auf den **vierten Donnerstag im November** fällt, ist der Black Friday ebenfalls ein "bewegliches Event" folglich immer der **vierte Freitag im November**.

| Jahr | Black Friday |
|---|---|
| 2020 | 27. November |
| 2021 | 26. November |
| 2022 | 25. November |
| 2023 | 24. November |
| 2026 | 27. November |
| 2027 | 26. November |
| 2028 | 24. November |

Diese Regel lässt sich programmatisch berechnen – wir müssen die Daten also **nicht** hart kodieren,
sondern können sie automatisch für beliebige Jahre ableiten.

In [2]:

# ── 2. Black Friday Umsatz-Effekt ($) ────────────────────────────────────────
def black_friday(year):
    return pd.date_range(f'{year}-11-01', f'{year}-11-30', freq='W-FRI')[3]

black_fridays = pd.to_datetime([black_friday(y) for y in [2020, 2021, 2022, 2023]])

rng = np.random.default_rng(seed=42)

df['bf_effect'] = 0
for bf in black_fridays:
    year_factor = np.clip(rng.normal(1.0, 0.2), 0.6, 1.4)

    for offset, extra_umsatz in [(-3, 18000), (-2, 22000), (-1, 30000), (0, 65000), (1, 35000)]:
        day_noise = rng.normal(0, extra_umsatz * 0.10)  # 10% tägliches Rauschen
        scaled = extra_umsatz * year_factor + day_noise
        df.loc[df['ds'] == bf + pd.Timedelta(days=offset), 'bf_effect'] = max(0, scaled)

# ── 3. Finale Zielvariable (y) berechnen ─────────────────────────────────────
# Prophet lernt auf der Spalte 'y'. Da wir ein additives Modell simulieren, 
# addieren wir einfach alle Bausteine zum täglichen Gesamtumsatz:
df['y'] = df['trend'] + df['noise'] + df['weekly'] + df['yearly'] + df['bf_effect']

# ── 4. Ausgaben zur Kontrolle ────────────────────────────────────────────────
print('✅ Synthetischer Umsatz-Datensatz erfolgreich generiert!')
print(f'   {len(df)} Tage vom {df["ds"].min().date()} bis {df["ds"].max().date()}')
print()
print('Eingebaute Muster im Überblick (Alles in USD):')
print(f'  Trend:        Wächst von 6.450 $ auf 19.350 $ pro Tag')
print(f'  Woche:        Samstag +2.580 $ / Dienstag -1.290 $')
print(f'  Jahr:         Amplitude ±3.225 $ (Sommer vs. Winter)')
print(f'  Black Friday: Bis zu +65.000 $ Extra-Umsatz am Peak-Tag')

✅ Synthetischer Umsatz-Datensatz erfolgreich generiert!
   1461 Tage vom 2020-01-01 bis 2023-12-31

Eingebaute Muster im Überblick (Alles in USD):
  Trend:        Wächst von 6.450 $ auf 19.350 $ pro Tag
  Woche:        Samstag +2.580 $ / Dienstag -1.290 $
  Jahr:         Amplitude ±3.225 $ (Sommer vs. Winter)
  Black Friday: Bis zu +65.000 $ Extra-Umsatz am Peak-Tag


C:\Users\betti\AppData\Local\Temp\ipykernel_14488\1004390100.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '17225.010095883063' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df['ds'] == bf + pd.Timedelta(days=offset), 'bf_effect'] = max(0, scaled)


---
## ✏️ Teil B: Aufgabe 2 – Prophet ohne Feiertage trainieren & plotten

Wir teilen den Datensatz in Training und Test auf:
- **Training (2020–2022):** Prophet lernt Trend und Saisonalität aus drei Jahren
- **Test (2023):** Wir prüfen, ob das Modell den Black Friday 2023 korrekt vorhersagt

```
─────────────────────────────────┬──────────────
        Training (2020–2022)     │  Test (2023)
─────────────────────────────────┴──────────────
                          01.01.2023
```

**Prophet API:**
```python
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(df_train[['ds', 'y']])          # Training
forecast = m.predict(df_test[['ds']]) # Vorhersage für Testzeitraum
# Ergebnis enthält: ds, yhat (Vorhersage), yhat_lower, yhat_upper
```

In [ ]:
# MUSTERLÖSUNG: Prophet ohne Feiertage
from prophet import Prophet  

print('Trainiere das Naive Modell (ignoriert Black Friday)...')


# --- Train/Test-Split ---
SPLIT_DATE = #### Datum ergaezen

df_train = df[df['ds'] < SPLIT_DATE]
df_test  = df[df['ds'] >= SPLIT_DATE]

print(f'Training: {df_train["ds"].min().date()} bis {df_train["ds"].max().date()} ({len(df_train)} Tage)')
print(f'Test:     {df_test["ds"].min().date()} bis {df_test["ds"].max().date()} ({len(df_test)} Tage)')

# 1. Modell initialisieren - bitte ausformulieren
m_naive = #####



SyntaxError: invalid syntax (3030640354.py, line 5)

In [ ]:
# Bitte plotte die Daten als überangeordnete Subplots mit der ganzen Zeitreihe im oberen Teil und mit Zoom auf diesen Bereich im unteren Teil

ZOOM_START = '2023-10-01'
ZOOM_END   = '2023-12-15'


---
## ✏️ Teil C: Aufgabe 3 – Black Friday-Kalender definieren

Prophet erwartet Feiertage als DataFrame mit folgenden Spalten:

| Spalte | Bedeutung | Beispiel |
|---|---|---|
| `holiday` | Name des Feiertags | `'Black Friday'` |
| `ds` | Datum(sliste) | `black_fridays` |
| `lower_window` | Tage **vor** dem Feiertag mit Effekt | `-3` = 3 Tage vorher |
| `upper_window` | Tage **nach** dem Feiertag mit Effekt | `1` = noch der Folgetag |

Wir haben in der Datengenerierung angenommen:
- **3 Tage vorher** starten Vorverkaufsangebote
- **1 Tag danach** laufen die Angebote noch weiter

Welche Werte für `lower_window` und `upper_window` bilden das korrekt ab?


In [ ]:
# --- BLACK FRIDAY KALENDER (vorgegeben) ---


In [ ]:
# Trainiere das Expertenmodell
print('Trainiere das Experten-Modell (kennt Black Friday)...')

# Gleicher Split wie beim naiven Modell – nur Trainingsdaten 2020–2022
##################### Bitte ausformulieren ###########################

######################################################################
print('✅ Experten-Modell trainiert!')

In [ ]:
# Bitte plotte beide Modellierung im Vergleich und zoome auf den interessanten Bereich um den Black Friday
ZOOM_START = '2023-10-01'
ZOOM_END   = '2023-12-15'


---
## 🏆 Zusammenfassung

| | Naives Modell | Experten-Modell |
|---|---|---|
| Black Friday erkannt? | ❌ Nein | ✅ Ja |
| Basisgeschäft korrekt? | ⚠️ Verzerrt | ✅ Sauber |
| Domänenwissen nötig? | ❌ | ✅ Analyst-in-the-Loop |

**Key Takeaway:** Prophet ist ein additives Modell – es zerlegt die Zeitreihe in Trend + Saisonalität + Feiertage. Feiertags-Wissen muss der Analyst explizit einbringen.

### 💬 Diskussionsfragen

1. **Was macht das naive Modell falsch?** Warum überschätzt es die Tage *rund um* den Black Friday?
2. **Was ist `lower_window` / `upper_window`?** Was würde passieren, wenn wir `upper_window=3` setzen?
3. **Wo liegen die Grenzen des Analyst-in-the-Loop-Ansatzes?** Was passiert bei einem neuen, unbekannten Event?
4. **Für welche Feiertage würdet ihr in einem echten E-Commerce-Datensatz noch Kalendereinträge anlegen?**